# Week 5: Text Generation using Vanilla RNN, LSTM, and GRU

**Aim:** Design and implement a Deep Learning model capable of learning the underlying structure, grammar, and contextual dependencies of a given text corpus to generate coherent and meaningful text sequences, using Vanilla RNN, LSTM, and GRU architectures.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import get_file

tf.random.set_seed(42)
np.random.seed(42)

## 1. Load Text Corpus

In [ ]:
path = get_file('shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(path, 'rb').read().decode(encoding='utf-8')
print(f"Corpus length: {len(text)} characters")
print(text[:300])

## 2. Preprocessing: Character-level Tokenization

In [ ]:
vocab = sorted(set(text))
vocab_size = len(vocab)
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])
print(f"Vocab size: {vocab_size}")

In [ ]:
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = sequences.map(split_input_target)

BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

## 3. Model Builders (Vanilla RNN / LSTM / GRU)

In [ ]:
embedding_dim = 256
rnn_units = 512

def build_model(cell_type, vocab_size, embedding_dim, rnn_units, batch_size):
    if cell_type == 'RNN':
        rnn_layer = SimpleRNN(rnn_units, return_sequences=True, stateful=True,
                               recurrent_initializer='glorot_uniform')
    elif cell_type == 'LSTM':
        rnn_layer = LSTM(rnn_units, return_sequences=True, stateful=True,
                          recurrent_initializer='glorot_uniform')
    elif cell_type == 'GRU':
        rnn_layer = GRU(rnn_units, return_sequences=True, stateful=True,
                         recurrent_initializer='glorot_uniform')
    else:
        raise ValueError("cell_type must be RNN, LSTM, or GRU")

    model = Sequential([
        Embedding(vocab_size, embedding_dim, batch_input_shape=[batch_size, None]),
        rnn_layer,
        Dense(vocab_size)
    ])
    return model

## 4. Train Each Model

In [ ]:
def loss_fn(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

EPOCHS = 5  # increase for better quality
histories = {}
models = {}

for cell_type in ['RNN', 'LSTM', 'GRU']:
    print(f"\n=== Training {cell_type} ===")
    model = build_model(cell_type, vocab_size, embedding_dim, rnn_units, BATCH_SIZE)
    model.compile(optimizer='adam', loss=loss_fn)
    history = model.fit(dataset, epochs=EPOCHS, verbose=1)
    models[cell_type] = model
    histories[cell_type] = history.history['loss']

## 5. Text Generation Function

In [ ]:
def build_gen_model(cell_type, weights):
    gen_model = build_model(cell_type, vocab_size, embedding_dim, rnn_units, batch_size=1)
    gen_model.set_weights(weights)
    gen_model.build(tf.TensorShape([1, None]))
    return gen_model

def generate_text(cell_type, start_string, num_generate=300, temperature=1.0):
    gen_model = build_gen_model(cell_type, models[cell_type].get_weights())
    input_eval = [char2idx[c] for c in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

    for layer in gen_model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

    for _ in range(num_generate):
        predictions = gen_model(input_eval)
        predictions = tf.squeeze(predictions, 0) / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)

## 6. Compare Generated Text Across Architectures

In [ ]:
start_string = "ROMEO: "

for cell_type in ['RNN', 'LSTM', 'GRU']:
    print(f"\n--- {cell_type} generated text ---")
    print(generate_text(cell_type, start_string))

## 7. Loss Comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
for cell_type, loss in histories.items():
    plt.plot(loss, label=cell_type)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss: Vanilla RNN vs LSTM vs GRU')
plt.legend()
plt.show()

## 8. Conclusion

- **Vanilla RNN** struggles with long-term dependencies due to vanishing gradients, often producing less coherent text over longer sequences.
- **LSTM** uses gating mechanisms (input, forget, output gates) to retain long-range context, generally yielding more coherent and grammatically consistent text.
- **GRU** simplifies LSTM's gating (update, reset gates) while achieving comparable performance with fewer parameters and faster training.

Overall, LSTM and GRU outperform vanilla RNN in capturing contextual dependencies for text generation tasks.